In [10]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
import sklearn
import torch
from torchvision import transforms
from google.colab import ai

In [11]:
from torch.utils.data import Dataset 
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

class FractureDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.paths)
    
    def __getitem__(self,index):
        img = Image.open(self.paths[index]).convert("RGB")
        img = self.transform(img)
        label = self.labels[index]
        return img, label

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
csv_path = "/content/drive/MyDrive/STINTSYMCO1/FracAtlas/dataset.csv"
df = pl.read_csv(csv_path)
df.head()


image_id,hand,leg,hip,shoulder,mixed,hardware,multiscan,fractured,fracture_count,frontal,lateral,oblique
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""IMG0000000.jpg""",0,1,0,0,0,0,1,0,0,1,1,0
"""IMG0000001.jpg""",0,1,0,0,0,0,1,0,0,1,1,0
"""IMG0000002.jpg""",0,1,0,0,0,0,1,0,0,1,1,0
"""IMG0000003.jpg""",0,1,0,0,0,0,1,0,0,0,1,1
"""IMG0000004.jpg""",0,1,0,0,0,0,1,0,0,0,1,1


In [14]:
fractured = df.filter(df["fractured"] == 1)
nonfractured = df.filter(df["fractured"] == 0)

train, test_validate = sklearn.model_selection.train_test_split(df, stratify=df["fractured"], train_size=0.7, test_size=0.3, random_state=42)
test, validation = sklearn.model_selection.train_test_split(test_validate, stratify=test_validate["fractured"], train_size=0.5, test_size=0.5, random_state=42)


train_paths = train.with_columns(
    pl.when(pl.col("fractured") == 1).then("/content/drive/MyDrive/STINTSYMCO1/FracAtlas/images/Fractured/" + pl.col("image_id")).otherwise("/content/drive/MyDrive/STINTSYMCO1/FracAtlas/images/Non_fractured/" + pl.col("image_id")).alias("paths")
)

test_paths = test.with_columns(
    pl.when(pl.col("fractured") == 1).then("/content/drive/<yDrove/STINTSYMCO1//FracAtlas/images/Fractured/" + pl.col("image_id")).otherwise("/content/drive/MyDrive/STINTSYMCO1/FracAtlas/images/Non_fractured/" + pl.col("image_id")).alias("paths")
)

validation_paths = validation.with_columns(
    pl.when(pl.col("fractured") == 1).then("/content/drive/MyDrive/STINTSYMCO1/FracAtlas/images/Fractured/" + pl.col("image_id")).otherwise("/content/drive/MyDrive/STINTSYMCO1/FracAtlas/images/Non_fractured/" + pl.col("image_id")).alias("paths")
)

train_paths = train_paths["paths"]
test_paths = test_paths["paths"]
validation_paths = validation_paths["paths"]

train_labels = train["fractured"]
test_labels = test["fractured"]
validation_labels = validation["fractured"]


In [15]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),                          
    transforms.Normalize([0.485, 0.456, 0.406],    
                         [0.229, 0.224, 0.225])   
])


In [16]:
from torch.utils.data import DataLoader

train_dataset = FractureDataset(train_paths.to_list(), train_labels.to_list(), transform=transform)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

val_dataset = FractureDataset(validation_paths.to_list(), validation_labels.to_list(), transform)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

test_dataset = FractureDataset(test_paths.to_list(), test_labels.to_list(), transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [17]:
from torchvision import models
import torch.nn as nn

model = models.mobilenet_v2(weights='IMAGENET1K_V1')
for param in model.parameters():
    param.requires_grad = False

model.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(model.last_channel, 1) 
)

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# pos_weight accounts for class imbalance (~4.7:1 non-fractured to fractured)
pos_weight = torch.tensor([3366 / 717]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Only train the classifier head — base is frozen
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-4)

In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
    # --- Training phase ---
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.float().to(device)  # BCEWithLogitsLoss expects float targets

        optimizer.zero_grad()
        outputs = model(images).squeeze(1)   # shape: (batch,) raw logits
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = (outputs > 0).long()         # logit > 0 means sigmoid > 0.5
        correct += (preds == labels.long()).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    # --- Validation phase ---
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.float().to(device)

            outputs = model(images).squeeze(1)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            preds = (outputs > 0).long()
            val_correct += (preds == labels.long()).sum().item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  "
          f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.4f}")
    
torch.save(model.state_dict(), 'finetunedNN.pth')
print('Model saved!')